# Jev, TypeSafe's System One model

A hands-on tour: what Jev is, a first API call, three things it's genuinely good at, and three popular ideas that don't hold up.

## 1. What it is

**Jev** is the first model from [TypeSafe AI](https://typesafe.ai/blog/introducing-system-one-models-and-jev), released on 15 September 2026 as the first of a new class they call *System One models* (after Kahneman's fast, intuitive "System 1" thinking). It is **not an LLM and never writes text.** You send it a `state` (a string or JSON) plus a set of typed questions: a **`noul`** (how likely is this yes/no statement true?), a **`choice`** (which of up to 255 options you define?), or a **`score`** (where on 2 to 10 ordered levels you define?). It answers every question in one parallel pass with a probability distribution, plus a `confidence` for choices and scores. Because an answer can only be one of the options you declared, it never returns malformed output or an invented label. That is what TypeSafe means by "can't hallucinate"; it can still pick the *wrong* valid option. It is trained for calibration (RLCD, *Reinforcement Learning for Calibrated Decisions*), so answers given 0.8 should be right about 80% of the time. The pitch is economics: 70 to 500 ms per call, $0.042 per million input tokens, output free. TypeSafe claims that is 40 to 200× faster than asking a frontier LLM the same question.

Think of it as a **smart `if` statement**: ordinary code owns the workflow and calls Jev for narrow, fuzzy judgments where hand-written rules are brittle, then acts on the probabilities (proceed, escalate to a human, or hand off to an LLM). The best rule of thumb we found comes from Theo: if a person with the information in front of them could answer in under about 10 seconds, Jev is probably good at it; *"if the task requires thinking, this model is not right"* ([t3.gg, 28:49](https://www.youtube.com/watch?v=F3YXg7AaKWE&t=28m49s)). TypeSafe's own [notes on jev-1.13's weak spots](https://docs.typesafe.ai/model-jaggedness/jev-1.13) list counting, arithmetic, date math, multi-hop questions, large states full of irrelevant detail, and adversarial input. Practical limits: text only (best in English), 64k tokens per request (32k for the state plus the longest question), and access through TypeSafe's API (waitlist), OpenRouter (used here), or the Vercel AI Gateway.

| Makes sense ✅ | Doesn't make sense ❌ |
|---|---|
| **Classify, route, triage**: tickets, emails, leads, and prompts (including routing prompts to a cheap or expensive LLM) | **Writing anything**: replies, summaries, code, "context compaction". There is no text output. |
| **Map-reduce over piles of text**: tag thousands of reviews, comments, or transcripts for cents | **Anything code computes exactly**: counting, arithmetic, dates, "did the tests pass?", path checks |
| **Real-time guardrails** in front of agents and LLMs: tool-call gates, jailbreak, PII, and policy checks | **Planning, lookahead, and control loops**: chess, video games, trading bots, "autopilot" |
| **Verifying another AI's output** against evidence you supply: claims vs. a policy, citations vs. a source | **Open-ended judging** ("which of these four LLM answers is best?") or fact-checking with no evidence in the state |
| **Feature extraction**: turn free text into probabilities for a dashboard or a downstream ML model | **Treating a probability as a fact**, or a classifier as a security boundary (it can be prompt-injected) |

**Sources:** the [launch post](https://typesafe.ai/blog/introducing-system-one-models-and-jev); the [TypeSafe docs](https://docs.typesafe.ai/introduction), especially [Jev with coding agents](https://docs.typesafe.ai/introduction/coding-agents) and the [jaggedness notes](https://docs.typesafe.ai/model-jaggedness/jev-1.13); 14 YouTube videos transcribed in `transcripts/` (see `transcripts/README.md`; timestamps below link to them); and write-ups from [Flavio Copes](https://flaviocopes.com/jev/), [eesel AI](https://www.eesel.ai/blog/typesafe-jev-review), and [Check Point Research](https://blog.checkpoint.com/ai-security/jev-is-not-a-language-model-but-it-breaks-like-one-prompt-injection-against-a-typed-decision-model/) (prompt injection).

## Run this notebook

From the repository root, run `uv sync`, open this notebook in Jupyter or VS Code, and select the project's `.venv` Python kernel. Set `OPENROUTER_KEY` in your shell or in a root `.env` file (the same name used by `hello_jev.py`). Each example makes a live, small, billable API request using synthetic data. Saved outputs are examples and may change on a later run; rerun the first example to see the response for the revised three-question request.

The request below copies the model, state, and questions from `hello_jev.py` and uses OpenRouter’s current `/api/alpha/decisions` endpoint. The authorization value is deliberately redacted in the printed request.

In [1]:
import json
import os
from pathlib import Path

import httpx
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
API_KEY = os.getenv("OPENROUTER_KEY")
if not API_KEY:
    raise RuntimeError("Set OPENROUTER_KEY in your environment or in a root .env file.")

URL = "https://openrouter.ai/api/alpha/decisions"
MODEL = "typesafe/jev-1.13"

def pretty(value):
    print(json.dumps(value, indent=2, ensure_ascii=False))

def ask_jev(state, questions):
    body = {"model": MODEL, "state": state, "questions": questions}
    response = httpx.post(
        URL,
        headers={"Authorization": f"Bearer {API_KEY}"},
        json=body,
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

## 1. The smallest call: full input and output

This is the request body from `hello_jev.py`. `is_urgent` asks for the probability that the message sounds urgent, `department` chooses from three teams, and `frustration` scores the customer’s tone on three ordered levels. The entire JSON request is printed below (with the secret omitted), followed by the HTTP status and complete response body. Notice that a Choice returns *all* option probabilities, not just the winner.

In [2]:
body = {
    "model": MODEL,
    "state": "Help! My payouts have been failing for 3 days.",
    "questions": {
        "is_urgent": {
            "type": "noul",
            "instructions": "Does this convey urgency?",
            "criteria": {"true": "Explicitly time-sensitive", "false": "No urgency expressed"},
        },
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "billing": "Payments, invoicing, refunds",
                "technical": "Bugs, outages, integrations",
                "sales": "Pricing, upgrades, new accounts",
            },
        },
        "frustration": {
            "type": "score",
            "instructions": "How frustrated is the customer?",
            "criteria": ["Calm", "Frustrated", "Very angry"],
        },
    },
}

print("INPUT (authorization secret redacted)")
pretty({
    "method": "POST",
    "url": URL,
    "headers": {"Authorization": "Bearer <redacted>", "Content-Type": "application/json"},
    "json": body,
})
response = httpx.post(
    URL,
    headers={"Authorization": f"Bearer {API_KEY}"},
    json=body,
    timeout=30,
)
print("\nHTTP STATUS:", response.status_code)
response.raise_for_status()
print("OUTPUT (complete response JSON)")
pretty(response.json())

INPUT (authorization secret redacted)
{
  "method": "POST",
  "url": "https://openrouter.ai/api/alpha/decisions",
  "headers": {
    "Authorization": "Bearer <redacted>",
    "Content-Type": "application/json"
  },
  "json": {
    "model": "typesafe/jev-1.13",
    "state": "Help! My payouts have been failing for 3 days.",
    "questions": {
      "is_urgent": {
        "type": "noul",
        "instructions": "Does this convey urgency?",
        "criteria": {
          "true": "Explicitly time-sensitive",
          "false": "No urgency expressed"
        }
      },
      "department": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {
          "billing": "Payments, invoicing, refunds",
          "technical": "Bugs, outages, integrations",
          "sales": "Pricing, upgrades, new accounts"
        }
      },
      "frustration": {
        "type": "score",
        "instructions": "How frustrated is the customer?",
        "criter

## 2. Good fit: rank the severity of a bug report

A support queue receives brief bug reports. A **Score** question rates one narrow dimension using concrete levels, while a **Noul** checks whether the report supplies reproduction steps. Code then decides whether the report needs human triage. This follows the transcript's support-triage idea ([IndyDevDan, about 03:20](https://www.youtube.com/watch?v=_U-O5lYhJ7Q&t=200s)) and TypeSafe's [Score guide](https://docs.typesafe.ai/primitives/score). The `0.55` threshold is only a demo policy.

In [3]:
bug_report = (
    "Export to PDF freezes the settings page in Safari. Steps: open Settings, "
    "click Export, then the spinner never stops. It works in Chrome, but 40 "
    "enterprise users cannot switch browsers because of company policy."
)
bug_result = ask_jev(bug_report, {
    "severity": {
        "type": "score",
        "instructions": "How severe is the reported product problem?",
        "criteria": [
            "Cosmetic issue; the feature still works",
            "A feature is broken, but the affected users have a practical workaround",
            "A feature is blocked for affected users and they have no practical workaround",
        ],
    },
    "has_steps": {
        "type": "noul",
        "instructions": "Does the report give concrete steps to reproduce the problem?",
    },
})
pretty(bug_result["answers"])
severity = bug_result["answers"]["severity"]
route = "human triage" if severity["confidence"] < 0.55 else (
    "priority bug queue" if severity["score"] >= 1.5 else "normal bug queue"
)
print("Queue:", route)
print("Ask for reproduction steps:", bug_result["answers"]["has_steps"]["noul"] < 0.5)

{
  "severity": {
    "type": "score",
    "score": 1.98,
    "legend": {
      "0": "Cosmetic issue; the feature still works",
      "1": "A feature is broken, but the affected users have a practical workaround",
      "2": "A feature is blocked for affected users and they have no practical workaround"
    },
    "probabilities": {
      "0": 0,
      "1": 0.02,
      "2": 0.98
    },
    "confidence": 0.96
  },
  "has_steps": {
    "type": "noul",
    "noul": 0.97
  }
}
Queue: priority bug queue
Ask for reproduction steps: False


## 3. Good fit: choose among known passages

Here the answer can only be one of four *existing* passage IDs. The Choice distribution ranks them; a separate Noul asks whether any passage actually answers the question. Code returns the original passage text, so Jev never has to invent a citation or passage. This pattern is related to the transcript's semantic “find in page” demo ([Matthew Berman, about 07:03](https://www.youtube.com/watch?v=jGD_UR4wMJc&t=423s)) and TypeSafe's [retrieval cookbook](https://docs.typesafe.ai/cookbooks/classifying_rag_passages).

In [4]:
query = "When does a meal expense need a manager's sign-off?"
passages = {
    "p1": "Vacation requests should be submitted two weeks before the first day off.",
    "p2": "A meal over $75 needs manager sign-off when the claim description does not clearly match the receipt.",
    "p3": "New employees must enroll in single sign-on before accessing company apps.",
    "p4": "Customer support tickets are assigned to the queue that owns the account.",
}
search_result = ask_jev(
    {"query": query, "passages": passages},
    {
        "best_passage": {
            "type": "choice",
            "instructions": "Which listed passage best answers the query?",
            "criteria": passages,
        },
        "answer_present": {
            "type": "noul",
            "instructions": "Does at least one listed passage directly answer the query?",
        },
    },
)
pretty(search_result["answers"])
found = search_result["answers"]["answer_present"]["noul"] >= 0.8
best = search_result["answers"]["best_passage"]
if found and best["confidence"] >= 0.5:
    print("Selected existing passage:", best["choice"], passages[best["choice"]])
else:
    print("No sufficiently clear passage; widen search or ask a person.")

{
  "best_passage": {
    "type": "choice",
    "choice": "p2",
    "probabilities": {
      "p4": 0,
      "p3": 0,
      "p1": 0,
      "p2": 1
    },
    "confidence": 1
  },
  "answer_present": {
    "type": "noul",
    "noul": 0.97
  }
}
Selected existing passage: p2 A meal over $75 needs manager sign-off when the claim description does not clearly match the receipt.


## 4. Good fit: mix semantic checks with a deterministic policy

The policy is: every claim needs a readable receipt; a meal above $75 needs manager sign-off if its description does not match the receipt. Jev answers the two fuzzy questions. Python handles the exact dollar threshold and the final branch. This mirrors TypeSafe's [workflow example](https://evals.typesafe.ai/) and avoids asking the model to do arithmetic or silently rewrite policy.

In [5]:
claim = {
    "submitted_category": "meal",
    "amount_usd": 82.00,
    "description": "Dinner with a client at Cafe Luna",
    "receipt_text": "Gadget Supply - USB-C dock - Total $82.00",
}
claim_result = ask_jev(claim, {
    "receipt_readable": {
        "type": "noul",
        "instructions": "Is the receipt text legible enough to identify a merchant, item, and total?",
    },
    "description_matches_receipt": {
        "type": "noul",
        "instructions": "Does the submitted description clearly match what the receipt says was purchased?",
    },
})
pretty(claim_result["answers"])
a = claim_result["answers"]
if a["receipt_readable"]["noul"] < 0.8:
    disposition = "ask for a clearer receipt"
elif claim["submitted_category"] == "meal" and claim["amount_usd"] > 75 and a["description_matches_receipt"]["noul"] < 0.8:
    disposition = "request manager sign-off"
else:
    disposition = "continue normal review"
print("Disposition:", disposition)

{
  "receipt_readable": {
    "type": "noul",
    "noul": 0.96
  },
  "description_matches_receipt": {
    "type": "noul",
    "noul": 0.02
  }
}
Disposition: request manager sign-off


## 5. Bad fit: treat a Choice as a personalized reply writer

Some discussions describe Jev as a replacement for a chat or coding model. It cannot write a new reply: the following call can only select one of two texts that *we* supplied. Neither text explains why the payout failed, since the state gives no cause. Selecting a preapproved template can be useful, but it does not fulfill a request to draft a tailored response ([Codevolution, about 04:05](https://www.youtube.com/watch?v=ZgXej_9isxY&t=245s); [Matthew Berman, about 02:13](https://www.youtube.com/watch?v=jGD_UR4wMJc&t=133s)).

In [6]:
customer_message = "My payout has failed for three days. Tell me why and when it will arrive."
canned_replies = {
    "acknowledge": "I'm sorry about the delay. We'll investigate your payout and follow up.",
    "self_service": "Please check your payout settings in the billing portal.",
}
reply_result = ask_jev(customer_message, {
    "reply": {
        "type": "choice",
        "instructions": "Which supplied response is the best next message?",
        "criteria": canned_replies,
    },
})
selected = reply_result["answers"]["reply"]["choice"]
print("Jev returned a choice:", selected)
print("Text supplied in advance:", canned_replies[selected])
print("New explanation of the failure returned by Jev:", reply_result.get("explanation"))

Jev returned a choice: acknowledge
Text supplied in advance: I'm sorry about the delay. We'll investigate your payout and follow up.
New explanation of the failure returned by Jev: None


## 6. Bad fit: use keep/drop labels as agent memory compaction

A proposed agent use case is to ask Jev which old messages to retain, then call that “compaction.” The runnable example below produces only keep/drop judgments. Compaction needs a faithful *synthesis* of the user's requirements, tool results, and causal links; filtering snippets cannot produce that summary. A long or incomplete state makes the mismatch worse. This concern is raised in the [Theo transcript, about 23:03](https://www.youtube.com/watch?v=F3YXg7AaKWE&t=1383s).

In [7]:
history = {
    "m1": "User: The export must be CSV, and account IDs must retain leading zeros.",
    "m2": "Assistant: I will inspect the current export code.",
    "m3": "Tool result: The CSV writer currently casts account IDs to integers.",
    "m4": "User: Please also include canceled accounts.",
}
memory_result = ask_jev(history, {
    key: {
        "type": "noul",
        "instructions": f"Should message {key} be retained when condensing this work history?",
    }
    for key in history
})
keep = [key for key in history if memory_result["answers"][key]["noul"] >= 0.5]
print("Keep/drop probabilities:")
pretty({key: value["noul"] for key, value in memory_result["answers"].items()})
print("Retained raw messages:", keep)
print("Synthesized implementation summary returned:", memory_result.get("summary"))

Keep/drop probabilities:
{
  "m1": 0.82,
  "m2": 0.5,
  "m3": 0.75,
  "m4": 0.72
}
Retained raw messages: ['m1', 'm2', 'm3', 'm4']
Synthesized implementation summary returned: None


## 7. Bad fit: approve a whole code change from a short snippet

There is interest in using Jev as an instant evaluator of agent or PR output. Narrow checks against complete evidence may be useful; “is this entire change correct, secure, and ready to merge?” is a different task. This call returns a probability even though it has no surrounding repository, tests, requirements, or deployment context. That number cannot establish readiness. See the [Theo transcript, about 22:20 and 28:15](https://www.youtube.com/watch?v=F3YXg7AaKWE&t=1340s) and TypeSafe's advice to ask [atomic questions](https://docs.typesafe.ai/primitives).

In [8]:
partial_review_state = {
    "claimed_goal": "Fix CSV export without changing account IDs",
    "patch_excerpt": "writer.writerow({'account_id': int(row['account_id'])})",
    "repository_context": "not supplied",
    "tests": "not supplied",
}
review_result = ask_jev(partial_review_state, {
    "ready_to_merge": {
        "type": "noul",
        "instructions": "Is this entire change correct, secure, and ready to merge?",
    },
})
pretty(review_result["answers"])
print("Evidence missing for a full review: repository context and test results.")
print("No merge action is taken from this probability.")

{
  "ready_to_merge": {
    "type": "noul",
    "noul": 0.14
  }
}
Evidence missing for a full review: repository context and test results.
No merge action is taken from this probability.


### What these examples show

Jev works best as a small decision inside a larger, testable program. Give it the evidence, constrain the answer to the shape you need, and let code apply exact rules or request review. A typed answer prevents an unexpected output format; it does not supply absent facts, generate a narrative, or prove a complex decision correct. The video transcripts in `transcripts/` are auto-generated captions, so their technical specifics should be checked against the [official API docs](https://docs.typesafe.ai/) and the live API.